# ID6004W Industrial Vision - Assignment 1
### Lectures 01-02: Image Filtering and Convolution

| | |
|---|---|
| **Released** | Oct 1st 2026 |
| **Due** | Oct 21st 2026 (5.00 PM IST) |
| **Total** | 100 marks (will be scaled down appropriately later) |

## Instructions

**Working in Colab.** Open this notebook in Google Colab and immediately use *File → Save a copy in Drive*; work only in your copy. Type your answers in the cells marked **Your answer**, directly below each question (double-click a cell to edit it). Mathematics can be typed in LaTeX between `$...$` or `$$...$$`. Photographs of hand-written working are **not** accepted in place of typed answers.

- **The writing must be your own.**
- Cells marked **Optional check** are not graded. Use them to verify a hand calculation numerically, but the answer you submit must be derived by hand with the steps shown.
-  **Submission.** When you are done, use *File → Download → Download .ipynb*, rename the file to `<ROLLNO>_assignment1.ipynb`, and upload it here: [SUBMISSION_LINK](https://forms.gle/pRwExrMB1dvKoU138). For queries contact Rishabh Shukla (da26e011@smail.iitm.ac.in). 
- **Late policy:** 10 marks will be deducted for each day the assignment is submitted late after the due date.

## Student details

- **Name:** Sohang Chopra
- **Roll number:** DA25M622

In [2]:
# Imports used by the optional check cells.
import numpy as np
from scipy import ndimage
rng = np.random.default_rng(0)

---
## Question 1. What is, and what is not, a convolution  [25 marks]

In Lecture 01 we proved that linearity together with shift invariance forces an operator to be a convolution. This question asks you to use that theorem as a *test*.

**(a)** For each operator $\mathcal{T}$ below, state whether it is (i) linear and (ii) shift invariant. If it is both, give its impulse response $h$ explicitly (as a small array, or as a formula). Here $f$ is the input image and $g=\mathcal{T}\{f\}$. **[15]**

1. $g(m,n) = f(m,n) - f(m-1,n)$
2. $g(m,n) = \operatorname{median}\big\{f(m+i,n+j) : |i|\le 1, |j|\le 1\big\}$
3. $g(m,n) = c(m,n)\,f(m,n)$, where $c$ is a fixed, non-constant mask (e.g. lens vignetting)
4. $g(m,n) = f(m,n)^{2}$
5. $g(m,n) = f(n,m)$ (transpose the image)
6. $g(m,n) = f(m,n) + 10$ (brighten)

**Your answer 1(a).**

* Linearity Test: $\forall a, b, g(a f_1 + b f_2) = a g(f_1) + b g(f_2)$
* Shift Invariance Test: $\forall m_0, n_0, g_{f_{shift}} = g_f(m - m_0, n - n_0) , \quad \text{where } f_{shift}(m,n) = f(m-m_0, n-n_0)$
* 2D Impulse Function $\delta(m,n)$ is $1$ at $(0,0)$, $0$ everywhere else.

1. $g(m,n) = f(m,n) - f(m-1,n)$

**Linearity Test:**

$$
g(a f_1(m,n) + b f_2(m,n)) = a f_1(m,n) + b f_2(m,n) - a f_1(m-1,n) + b f_2(m-1,n) \\
\implies g(a f_1(m,n) + b f_2(m,n)) = a (f_1(m,n) - f_1(m-1,n)) + b (f_2(m,n) - f_2(m-1,n)) \\
\implies g(a f_1(m,n) + b f_2(m,n)) = a g(f_1(m,n)) + b g(f_2(m,n))
$$

Hence $g$ is *linear* .

**Shift Invariance Test:**

$$
f_{shift}(m,n) = f(m-m_0, n-n_0) \\
g_{f_{shift}} = f(m-m_0, n-n_0) - f(m-m_0-1,n-n_0) = g_f(m-m_0, n-n_0)
$$

Hence $g$ is *shift invariant* .

**Impulse Response**: Taking $f(m,n) = \delta(m,n)$ :

$$
g(m,n) = \delta(m,n) - \delta(m-1,n) \\
g(0,0) = \delta(0,0) - \delta(-1,0) = 1 - 0 = 1 \\
g(1,0) = \delta(1,0) - \delta(0,0) = 0 - 1 = -1 \\
$$

For any other $(m,n)$, $g(m,n)$ is 0, so these 2 are the only non-zero responses.

*Impulse Response Function* is:

$$
g(m,n) = \begin{cases} 
   1 & \text{if } m = 0 && n = 0 \\ 
  -1 & \text{if } m = 1 && n = 0 \\
   0 & \text{otherwise}
\end{cases}
$$

*Impulse Response Array* (i.e. out of infinite matrix, keeping only $m,n$ positions with non-zero responses) is $\begin{pmatrix} 1 \\ -1 \end{pmatrix}$ .

2. $g(m,n) = \operatorname{median}\big\{f(m+i,n+j) : |i|\le 1, |j|\le 1\big\}$

**Linearity**:

Consider adding two images whose neighbourhoods each contain five zeros and four ones, with their ones in complementary positions. Their individual medians are $0$, but their sum has a median of $1$. 

Therefore, this counter-example shows that $g$ is *not linear* .

**Shift Invariance**:

The median is computed over the same relative $3 \times 3$ neighbourhood at every pixel. Shifting the input image shifts the output by the same amount.

Hence, the operator is *shift invariant* .

3. $g(m,n) = c(m,n) f(m,n)$, where $c$ is a fixed, non-constant mask (e.g. lens vignetting)

**Linearity**:

$$ g_{a f_1 + b f_2}(m,n) = c(m,n) (a f_1(m,n) + b f_2(m,n)) = a c(m,n) f_1(m,n) + b c(m,n) f_2(m,n) = a g_1(m,n) + b g_2(m,n) $$

Hence $g$ is *linear* .

**Shift Invariance**:

$$
f_{shift}(m,n) = f(m-m_0, n-n_0) \\
g_{f_shift}(m,n) = c(m,n) f(m-m_0, n-n_0) \\
g_f(m-m_0, n-n_0) = c(m-m_0, n-n_0) f(m-m_0, n-n_0)
$$

$g_{f_shift}(m,n) \neq g_f(m-m_0, n-n_0)$, so $g$ is *not shift invariant* .

4. $g(m,n) = f(m,n)^{2}$
   
**Linear**: $g$ is *not linear* as input image $f$ is squared.

**Shift Invariant**: $g$ is *not shift invariant* due to square operation.

5. $g(m,n) = f(n,m)$ (transpose the image)

**Linear**: $g$ is *linear* as transpose is a linear operation.

**Shift Invariant**: 

$$
f_{shift}(m,n) = f(m-m_0, n-n_0) \\
g_{f_{shift}}(m,n) = f(n-m_0, m-n_0) \\
g_f(m-m_0, n-n_0) = f(n-n_0, m-m_0)
$$

$g_{f_shift}(m,n) \neq g_f(m-m_0, n-n_0)$, so $g$ is *not shift invariant* .

6. $g(m,n) = f(m,n) + 10$ (brighten)

**Linearity**:

$$
g_{a f_1 + b f_2}(m,n) = a f_1(m,n) + b f_2(m,n) + 10 (a + b) \\
a g_1(m,n) + b g_2(m,n) = a f_1(m,n) + b f_2(m,n)
$$

$g(a f_1 + b f_2) \neq a g(f_1) + b g(f_2)$, so $g$ is *not linear* .

**Shift Invariant**:

$$
f_{shift}(m,n) = f(m-m_0, n-n_0) \\
g_{f_{shift}}(m,n) = f(m-m_0, n-n_0) + 10 = g_f(m-m_0, n-n_0)
$$

Hence $g$ is *not shift invariant* .

In [4]:
# Optional check for 1(a). The six operators, with periodic boundaries so that shifts are exact.
N = 16
c_mask = rng.uniform(0.5, 1.0, (N, N))          # a fixed, non-constant mask

T = {
    1: lambda f: f - np.roll(f, 1, axis=0),
    2: lambda f: ndimage.median_filter(f, size=3, mode="wrap"),
    3: lambda f: c_mask * f,
    4: lambda f: f ** 2,
    5: lambda f: f.T,
    6: lambda f: f + 10,
}

def shift(f, dm, dn):
    return np.roll(f, (dm, dn), axis=(0, 1))

def is_linear(op):
    f1 = rng.random((N, N))
    f2 = rng.random((N, N))
    a = rng.uniform(-1, 1)
    b = rng.uniform(-1, 1)
    
    left_side = op(a * f1 + b * f2)
    right_side = a * op(f1) + b * op(f2)
    
    return np.allclose(left_side, right_side)

def is_shift_invariant(op):
    f = rng.random((N, N))
    dm = rng.integers(-N//2, N//2)
    dn = rng.integers(-N//2, N//2)
    
    left_side = op(shift(f, dm, dn))
    right_side = shift(op(f), dm, dn)
    
    return np.allclose(left_side, right_side)

for op_id, op in T.items():
    linear = is_linear(op)
    shift_invariant = is_shift_invariant(op)
    print(f"Operator {op_id}: Linear: {linear}, Shift Invariant: {shift_invariant}")

Operator 1: Linear: True, Shift Invariant: True
Operator 2: Linear: False, Shift Invariant: True
Operator 3: Linear: True, Shift Invariant: False
Operator 4: Linear: False, Shift Invariant: True
Operator 5: Linear: True, Shift Invariant: False
Operator 6: Linear: False, Shift Invariant: True


**(b)** Let $\mathcal{T}_1$ and $\mathcal{T}_2$ be two LSI systems with impulse responses $h_1$ and $h_2$. Show that the cascade $\mathcal{T}_2\{\mathcal{T}_1\{f\}\}$ is again LSI, identify its impulse response, and state why the order of the two systems does not matter. **[10]**

**Your answer 1(b).**

$$
f' = T_2\{T_1\{f\}\} \\
g_{a f_1' + b f_2'} = a T_2\{T_1\{f_1'\}\} + b T_2\{T_1\{f_2'\}\} = a g_1 + b g_2 , \quad \text{Linearity} \\
f_{shift}'(m,n) = f'(m-m_0, n-n_0)
g_{f_{shift}'}(m,n) = T_2\{T_1\{f_{shift}(m-m_0, n-n_0)\}\} = g_{f'}(m-m_0, n-n_0) , \quad \text{Shift Invariance}
$$

Hence proved that $T_2\{T_1\{f\}\}$ is *linear and shift invariant* .

Impulse Response (note that $f \ast g$ represents convolution):

$$
h'(m,n) = T_2\{T_1\{\delta(m,n)\}\} = T_2\{h_1(m,n)\} = (f'(m,n) \ast h_1(m,n)) \ast h_2(m,n) = f'(m,n) \ast (h_1(m,n) \ast h_2(m,n)) \\
\implies h'(m,n) = h_1(m,n) \ast h_2(m,n) , \quad (\text{since } h'(m,n) = f(m,n) \ast h'(m,n))
$$

Since convolution is commutative, so $h'(m,n) = h_1(m,n) \ast h_2(m,n) = h_2(m,n) \ast h_1(m,n)$ . So order of operations $T_1$ and $T_2$ doesn't matter.

---
## Question 2. Separability and rank  [25 marks]

**(a)** Determine the rank of each kernel below, and for those of rank one give an explicit factorisation $H=h_1h_2^{\top}$. Do this by hand, using the fact that a nonzero matrix has rank one exactly when every $2\times2$ minor vanishes. **[15]**

$$
A=\frac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}
\quad
B=\begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}
\quad
C=\begin{bmatrix}1&2&3\\2&4&6\\3&6&9\end{bmatrix}
\quad
D=\begin{bmatrix}1&1&1\\1&2&1\\1&1&1\end{bmatrix}
$$

**Your answer 2(a).**

Doing row reductions on all the matrices to find ranks:

1. 

$$
A=\frac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix} \\
A = \frac{1}{16} \begin{bmatrix} 1 & 2 & 1 \\ 0 & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix} , \quad (R_2 -> R_2 - 2 R_1 , R_3 -> R_3 - R_1)
$$

So $A$ matrix has rank 1.

A solution for $A = h_1 h_2^T$ can be found therefore by simply using first non-zero row and column vectors of A (scaled by 1/4 each that multiplies to 1/16):

$$ h_1 = \frac{1}{4} \begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix}, \quad h_2 = \frac{1}{4} \begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix} $$

2. 

$$
B=\begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix} \\
B = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 0 & 0 \end{bmatrix} , \quad (R_3 \rightarrow R_3 - R_1)
$$

So $B$ matrix has rank 2.

3. 

$$
C=\begin{bmatrix}1&2&3\\2&4&6\\3&6&9\end{bmatrix} \\
C = \begin{bmatrix} 1 & 2 & 3 \\ 0 & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix} , \quad (R_2 \rightarrow R_2 - 2 R_1, R_3 \rightarrow R_3 - 3 R_1)
$$

So $C$ matrix has rank 1. A solution for $B = h_1 h_2^T$ is:

$$ h_1 = \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix} , \quad h_2 = \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix} $$

4. 

$$
D=\begin{bmatrix}1&1&1\\1&2&1\\1&1&1\end{bmatrix} \\
D = \begin{bmatrix} 1 & 1 & 1 \\ 0 & 2 & 0 \\ 0 & 0 & 0 \end{bmatrix} , \quad (R_2 \rightarrow R_2 - R_1, R_3 \rightarrow R_3 - R_1)
$$

So $D$ matrix has rank 2.

In [10]:
# Optional check for 2(a).
A = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16
B = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])
C = np.array([[1, 2, 3], [2, 4, 6], [3, 6, 9]])
D = np.array([[1, 1, 1], [1, 2, 1], [1, 1, 1]])

hA1 = hA2 = 1/4 * np.array([[1], [2], [1]])
hC1 = hC2 = np.array([[1], [2], [3]])

print("Rank of A:", np.linalg.matrix_rank(A), f"conv(h1,h2) = A is {(hA1 @ hA2.T == A).all()}")
print("Rank of B:", np.linalg.matrix_rank(B))
print("Rank of C:", np.linalg.matrix_rank(C), f"conv(h1,h2) = C is {(hC1 @ hC2.T == C).all()}")
print("Rank of D:", np.linalg.matrix_rank(D))

Rank of A: 1 conv(h1,h2) = A is True
Rank of B: 2
Rank of C: 1 conv(h1,h2) = C is True
Rank of D: 2


**(b)** An $N\times N$ image is smoothed with a Gaussian of $\sigma=5$, truncated to $2\lceil 3\sigma\rceil+1$ taps. State the kernel size $M$, the number of multiplications per pixel for a direct 2D pass and for two 1D passes, and the resulting speed-up. **[10]**

**Your answer 2(b).**

The 1D kernel size $M$ (number of taps) for a truncated Gaussian with $\sigma = 5$ is given by:

$$M = 2\lceil 3\sigma \rceil + 1$$

Substituting $\sigma = 5$:

$$3\sigma = 15 \implies \lceil 15 \rceil = 15$$

$$M = 2(15) + 1 = \mathbf{31}$$

Thus, the spatial filter is an **$M \times M = 31 \times 31$** kernel.

Because a Gaussian kernel is **2D separable** ($G(x,y) = G_1(x) \cdot G_1(y)$), we can compare direct 2D convolution against two sequential 1D filtering passes (horizontal then vertical).

* **Direct 2D Pass:**
Convolving an image with an $M \times M$ non-separable 2D spatial kernel requires $M^2$ multiplications per pixel.

$$\text{Multiplications (2D)} = M^2 = 31^2 = \mathbf{961 \text{ multiplications/pixel}}$$


* **Two 1D Passes (Separable):**
Applying a 1D filter of length $M$ horizontally requires $M$ multiplications per pixel, and applying the 1D filter vertically requires another $M$ multiplications per pixel.

$$\text{Multiplications (Separable)} = 2M = 2 \times 31 = \mathbf{62 \text{ multiplications/pixel}}$$

The theoretical speed-up ratio $S$ in terms of multiplication computational complexity is:

$$S = \frac{\text{Multiplications (2D)}}{\text{Multiplications (Separable)}} = \frac{M^2}{2M} = \frac{M}{2}$$

Substituting $M = 31$:

$$S = \frac{31}{2} = \mathbf{15.5}$$

---
## Question 3. Noise gain  [25 marks]

Let the noise be i.i.d. with variance $\sigma_\eta^{2}$, so that a linear filter $w$ produces output noise variance $\sigma_\eta^{2}\sum_k w_k^{2}$.

**(a)** Compute $\sum_k w_k^{2}$ for each of the four kernels below, and give the output noise standard deviation as a multiple of $\sigma_\eta$. **[10]**

$$
\text{box } \tfrac19\begin{bmatrix}1&1&1\\1&1&1\\1&1&1\end{bmatrix}
\quad
\text{Gaussian } \tfrac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}
\quad
\text{Sobel } \begin{bmatrix}1&0&-1\\2&0&-2\\1&0&-1\end{bmatrix}
\quad
\text{sharpen } \begin{bmatrix}0&-1&0\\-1&5&-1\\0&-1&0\end{bmatrix}
$$

**Your answer 3(a).**

$$\sigma_{\text{out}}^2 = \sigma_\eta^2 \sum_k w_k^2 \implies \sigma_{\text{out}} = \sqrt{\sum_k w_k^2} \cdot \sigma_\eta$$

1. Box Filter

$$W = \frac{1}{9} \begin{bmatrix}1&1&1\\1&1&1\\1&1&1\end{bmatrix}$$

* **Sum of squared weights ($\sum_k w_k^2$):**

$$\sum_k w_k^2 = 9 \times \left(\frac{1}{9}\right)^2 = \frac{9}{81} = \frac{1}{9} \approx 0.1111$$

* **Output noise standard deviation ($\sigma_{\text{out}}$):**

$$\sigma_{\text{out}} = \sqrt{\frac{1}{9}} \sigma_\eta = \mathbf{\frac{1}{3} \sigma_\eta \approx 0.3333 \sigma_\eta}$$


2. Gaussian Filter

$$W = \frac{1}{16} \begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}$$

* **Sum of squared weights ($\sum_k w_k^2$):**

$$\sum_k w_k^2 = \left(\frac{1}{16}\right)^2 \Big(4(1^2) + 4(2^2) + 4^2\Big) = \frac{1}{256} (4 + 16 + 16) = \frac{36}{256} = \frac{9}{64} = 0.140625$$

* **Output noise standard deviation ($\sigma_{\text{out}}$):**

$$\sigma_{\text{out}} = \sqrt{\frac{9}{64}} \sigma_\eta = \mathbf{\frac{3}{8} \sigma_\eta = 0.375 \sigma_\eta}$$


3. Sobel Filter

$$W = \begin{bmatrix}1&0&-1\\2&0&-2\\1&0&-1\end{bmatrix}$$

* **Sum of squared weights ($\sum_k w_k^2$):**

$$\sum_k w_k^2 = (1^2 + (-1)^2) + (2^2 + (-2)^2) + (1^2 + (-1)^2) = 2 + 8 + 2 = 12$$

* **Output noise standard deviation ($\sigma_{\text{out}}$):**

$$\sigma_{\text{out}} = \sqrt{12} \sigma_\eta = \mathbf{2\sqrt{3} \sigma_\eta \approx 3.4641 \sigma_\eta}$$


4. Sharpen Filter

$$W = \begin{bmatrix}0&-1&0\\-1&5&-1\\0&-1&0\end{bmatrix}$$

* **Sum of squared weights ($\sum_k w_k^2$):**

$$\sum_k w_k^2 = 5^2 + 4(-1)^2 = 25 + 4 = 29$$

* **Output noise standard deviation ($\sigma_{\text{out}}$):**

$$\sigma_{\text{out}} = \sqrt{29} \sigma_\eta = \mathbf{\sqrt{29} \sigma_\eta \approx 5.3852 \sigma_\eta}$$

In [16]:
box = (1/9) * np.ones((3,3))
gauss = (1/16) * np.array([[1,2,1],[2,4,2],[1,2,1]])
sobel = np.array([[1,0,-1],[2,0,-2],[1,0,-1]])
sharpen = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])

kernels = {
    "Box": box,
    "Gaussian": gauss,
    "Sobel": sobel,
    "Sharpen": sharpen
}

for name, k in kernels.items():
    sum_w2 = np.sum(k**2)
    std_mult = np.sqrt(sum_w2)
    print(f"{name}: sum(w_k^2) = {sum_w2:.6f} ({sum_w2}), std_dev = {std_mult:.6f} * sigma_eta")

Box: sum(w_k^2) = 0.111111 (0.1111111111111111), std_dev = 0.333333 * sigma_eta
Gaussian: sum(w_k^2) = 0.140625 (0.140625), std_dev = 0.375000 * sigma_eta
Sobel: sum(w_k^2) = 12.000000 (12), std_dev = 3.464102 * sigma_eta
Sharpen: sum(w_k^2) = 29.000000 (29), std_dev = 5.385165 * sigma_eta


**(b)** Prove that among *all* $3\times3$ kernels whose weights sum to $1$, the box kernel minimises $\sum_k w_k^{2}$. (One line of Cauchy-Schwarz.) **[5]**

**Your answer 3(b).**

By the **Cauchy–Schwarz inequality** for vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^N$:

$$(\mathbf{u} \cdot \mathbf{v})^2 \le \Vert{}\mathbf{u}\Vert{}^2 \Vert{}\mathbf{v}\Vert{}^2$$

Setting $\mathbf{u} = (w_1, w_2, \dots, w_9)^T$ as the vector of the 9 kernel weights and $\mathbf{v} = (1, 1, \dots, 1)^T$ as a vector of ones:

$$\left(\sum_{k=1}^{9} w_k \cdot 1\right)^2 \le \left(\sum_{k=1}^{9} w_k^2\right) \left(\sum_{k=1}^{9} 1^2\right) \implies 1^2 \le 9 \sum_{k=1}^{9} w_k^2 \implies \sum_{k=1}^{9} w_k^2 \ge \frac{1}{9}$$

Since the box kernel $w_k = \frac{1}{9}$ for all $k$ achieves $\sum_k w_k^2 = \frac{1}{9}$ (equality holds when $\mathbf{u}$ is proportional to $\mathbf{v}$), it minimizes $\sum_k w_k^2$. Hence proved.

**(c)** Your answers to (a) and (b) together say that the box filter is a strictly better denoiser, per pixel of support, than the $3\times3$ Gaussian. Yet the Gaussian is mostly the preferred choice. Give one distinct reason why. **[5]**

**Your answer 3(c).**

While the box filter minimizes output noise variance, the **Gaussian filter is generally preferred** for two key reasons in image processing:

1. Spatial Rotational Symmetry (Isotropy)

* **Gaussian Filter:** Is rotationally symmetric (isotropic). It smoothes equally in all directions, so image features, edges, and orientations are treated uniformly regardless of direction.
* **Box Filter:** Is rectangularly asymmetric. It biases smoothing along horizontal and vertical axes, introducing directional artifacts (such as turning sharp corners into axis-aligned squares).

2. Frequency Domain Properties (No Ringing / Phase Artifacts)

* **Gaussian Filter:** The Fourier transform of a Gaussian is another Gaussian. It has no negative side lobes in the frequency domain, meaning it acts as an ideal low-pass filter without introducing **phase reversals or ringing artifacts (sinc-like ripples)** near sharp edges.
* **Box Filter:** Its Fourier transform is a 2D **sinc function** with prominent side lobes. These side lobes cause high-frequency oscillations (ringing) and phase inversion across high-contrast edges in the output image.

**(d)** An alternative to spatial smoothing is to average $L$ frames of a static scene. How many frames are needed to match the noise reduction of one $3\times3$ box filter, and what does frame averaging give you that the box filter cannot? **[5]**

**Your answer 3(d).**

1. Number of Frames Needed

When averaging $L$ independent frames $I_1, I_2, \dots, I_L$ of a static scene, the variance of the temporal average is given by:

$$\sigma_{\text{avg}}^2 = \frac{\sigma_\eta^2}{L}$$

From part **(a)**, applying a single $3\times3$ box filter yields an output noise variance of:

$$\sigma_{\text{box}}^2 = \frac{1}{9} \sigma_\eta^2$$

Equating the two noise variances:

$$\frac{\sigma_\eta^2}{L} = \frac{\sigma_\eta^2}{9} \implies \mathbf{L = 9 \text{ frames}}$$

Thus, averaging **9 frames** temporally achieves the exact same noise reduction as applying a single $3\times3$ spatial box filter.

2. Advantage of Frame Averaging Over Spatial Smoothing

Frame averaging preserves **full spatial resolution and sharp edge detail**.

* **Spatial Box Filtering:** Combines neighboring pixels across space, which blurs sharp edges, fine textures, and fine structural details in the image.
* **Temporal Frame Averaging:** Averages pixels over time *at the exact same spatial location*. Because the scene is static, it reduces random zero-mean noise without causing any spatial blur or loss of resolution.

---
## Question 4. Zero-crossings  [25 marks]

**(a)** Show that the stencil $[1\;\;{-}2\;\;1]$ annihilates every affine ramp $f(x)=ax+b$, and that the five-point $\nabla^{2}$ annihilates every plane $f(x,y)=ax+by+c$. What does this tell you about the response of $\nabla^{2}$ inside a smoothly shaded region, and why is that desirable? **[15]**

**Your answer 4(a).**

A 1D discrete stencil $[1\;\;{-}2\;\;1]$ applied to a function $f(x)$ at point $x$ evaluates as:

$$\mathcal{S}[f](x) = 1 \cdot f(x - 1) - 2 \cdot f(x) + 1 \cdot f(x + 1)$$

Substitute the affine function $f(x) = ax + b$:

$$\begin{aligned} \mathcal{S}[f](x) &= [a(x - 1) + b] - 2(ax + b) + [a(x + 1) + b] \\ &= (ax - a + b) - (2ax + 2b) + (ax + a + b) \\ &= (ax + ax - 2ax) + (-a + a) + (b + b - 2b) \\ &= 0 \end{aligned}$$

*Thus, the 1D second-difference stencil annihilates every 1D affine ramp $f(x) = ax + b$.*

The standard 5-point discrete Laplacian operator applied to a 2D grid at $(x, y)$ is defined as:

$$\nabla^2 f(x, y) = f(x+1, y) + f(x-1, y) + f(x, y+1) + f(x, y-1) - 4f(x,y)$$

Substitute the 2D affine planar equation $f(x, y) = ax + by + c$:

$$\begin{aligned} \nabla^2 f(x,y) &= [a(x+1) + by + c] + [a(x-1) + by + c] \\ &\quad + [ax + b(y+1) + c] + [ax + b(y-1) + c] \\ &\quad - 4(ax + by + c) \\ &= (ax + a + by + c) + (ax - a + by + c) \\ &\quad + (ax + by + b + c) + (ax + by - b + c) \\ &\quad - 4ax - 4by - 4c \\ &= 4ax + 4by + 4c - 4ax - 4by - 4c \\ &= 0 \end{aligned}$$

*Thus, the 5-point discrete Laplacian annihilates every 2D linear/affine plane $f(x, y) = ax + by + c$.*

Smoothly shaded regions in images (such as gradients caused by linear illumination changes across a surface) can be modeled locally by an affine intensity function $f(x, y) = ax + by + c$.

Since $\nabla^2 f(x, y) = 0$ for all linear intensity variations, *the response of the Laplacian operator inside these smoothly shaded regions is 0*.

#### Why This Is Desirable

1. **Invariance to Linear Shading & Illumination:** Edge detectors (such as Marr-Hildreth or zero-crossing detectors) search for rapid intensity transitions (edges) where zero-crossings of $\nabla^2 f$ occur. If smooth gradients produced non-zero outputs, shading variations would introduce spurious edge detections.
2. **Selective Feature Detection:** By yielding zero response to constant background intensities ($a=b=0$) as well as steady linear gradients ($a, b \neq 0$), $\nabla^2$ isolates **true structural discontinuities** (step edges, lines, and corners) where intensity slope changes abruptly (i.e., non-zero second derivatives).

In [15]:
# Define the stencils
stencil_1d = np.array([1, -2, 1])
laplacian_5pt = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])

# Define coefficients for the affine ramp and plane
a, b, c = 3.0, 5.0, 2.0

# 1D Affine ramp: f(x) = a*x + b
x = np.arange(20.0)
ramp = a * x + b

# 2D Plane: f(x, y) = a*X + b*Y + c
X, Y = np.meshgrid(np.arange(20.0), np.arange(20.0))
plane = a * X + b * Y + c

# Apply convolution and slice the interior [1:-1] to exclude boundary padding effects
stencil_result = ndimage.convolve1d(ramp, stencil_1d, mode="constant")[1:-1]
laplacian_result = ndimage.convolve(plane, laplacian_5pt, mode="constant")[
    1:-1, 1:-1
]

print("Stencil result (interior):", stencil_result)
print("All close to zero?", np.allclose(stencil_result, 0))
print("\nLaplacian result (interior):\n", laplacian_result)
print("All close to zero?", np.allclose(laplacian_result, 0))

Stencil result (interior): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
All close to zero? True

Laplacian result (interior):
 [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0.

**(b)** A 1D edge is sampled and the second difference takes the values

$$f''(62)=+1.07,\qquad f''(63)=+0.33,\qquad f''(64)=-0.49 .$$

Locate the zero-crossing to sub-pixel accuracy by linear interpolation, showing the arithmetic. The corresponding $|f'|$ values are $14.10,\,14.80,\,14.72$; state the best estimate of the edge location obtainable from $\arg\max|f'|$ alone. **[10]**

**Your answer 4(b).**

**Zero-Crossing Location by Linear Interpolation**:

To locate the zero-crossing of $f''$, we find the interval where $f''$ changes sign:

* $f''(63) = +0.33$
* $f''(64) = -0.49$

Since $f''$ changes sign between $x = 63$ and $x = 64$, the zero-crossing lies within the interval $[63, 64]$.

Using linear interpolation between $(x_1, y_1) = (63, 0.33)$ and $(x_2, y_2) = (64, -0.49)$:

$$\text{Slope } m = \frac{f''(64) - f''(63)}{64 - 63} = \frac{-0.49 - 0.33}{1} = -0.82$$

Setting $f''(x) = 0$:

$$0 - f''(63) = m \cdot (x - 63)$$

$$-0.33 = -0.82 \cdot (x - 63)$$

$$x - 63 = \frac{-0.33}{-0.82} = \frac{33}{82} \approx 0.4024$$

$$x \approx 63.4024 \quad (\text{or } 63.40)$$

**Edge Location from $\arg\max\vert{}f'\vert{}$ Alone**:

Comparing the discrete magnitude of the first derivative at each pixel location:

* $\vert{}f'(62)\vert{} = 14.10$
* $\vert{}f'(63)\vert{} = 14.80$
* $\vert{}f'(64)\vert{} = 14.72$

The maximum value of $\vert{}f'\vert{}$ occurs at $x = 63$ with $\vert{}f'(63)\vert{} = 14.80$.

Therefore, the best estimate of the edge location using $\arg\max\vert{}f'\vert{}$ alone (at pixel resolution) is:

$$x = 63$$

In [14]:
# Optional check for 4(b).
# Linear interpolation between (63, 0.33) and (64, -0.49)
x1, y1 = 63, 0.33
x2, y2 = 64, -0.49

# Line equation: y - y1 = m * (x - x1)
# 0 - y1 = m * (x_zero - x1) => x_zero = x1 - y1 / m
m = (y2 - y1) / (x2 - x1)
x_zero = x1 - y1 / m

print(f"Slope m: {m}")
print(f"Zero crossing x: {x_zero}")

Slope m: -0.8200000000000001
Zero crossing x: 63.40243902439025


---
*End of Assignment 1.*